# Package  and module require

In [ ]:
import os
import shutil
import requests
import xml.etree.ElementTree as ET
from pathlib import Path
from tqdm.auto import tqdm
from requests.auth import HTTPBasicAuth

# CONFIGURATION

In [ ]:
CONSUMER_KEY = "au6LAHeBprVaxvOsF6sLV2klkgwa"
CONSUMER_SECRET = "6QDGD9eCqm9ueWVnKbu5fkJoYs4a"

METALINK_FILE = "C:/Users/LENOVO/Downloads/cart-SEYDOU.xml"

OUTPUT_DIR = Path("downloader/outputs/netcdf/sarah3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR = 1983
END_YEAR = 2025

TOKEN_URL = "https://api.eumetsat.int/token"

# AUTHENTIFICATION

In [ ]:
# AUTHENTIFICATION

print("token obtained...")

response = requests.post(
    TOKEN_URL,
    auth=HTTPBasicAuth(CONSUMER_KEY, CONSUMER_SECRET),
    data={"grant_type": "client_credentials"},
)

response.raise_for_status()

ACCESS_TOKEN = response.json()["access_token"]

headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}"
}

print("Authentification succes.\n")

# METALINK READING

tree = ET.parse(METALINK_FILE)
root = tree.getroot()

namespace = {
    "ml": "urn:ietf:params:xml:ns:metalink"
}

files = root.findall("ml:file", namespace)

print(f"{len(files)} produits trouvés dans le Metalink.\n")

downloaded = 0
skipped = 0
failed = 0

# DOWLOADING

In [2]:
# DOWLOADING

for item in tqdm(files):

    filename = item.attrib["name"]

    try:
        year = int(filename[5:9])
    except Exception:
        continue

    if year < START_YEAR or year > END_YEAR:
        continue

    outfile = OUTPUT_DIR / filename

    if outfile.exists():
        skipped += 1
        continue

    url = item.find("ml:url", namespace).text.strip()

    try:

        r = requests.get(url, headers=headers, stream=True)

        r.raise_for_status()

        with open(outfile, "wb") as f:

            for chunk in r.iter_content(chunk_size=1024 * 1024):

                if chunk:
                    f.write(chunk)

        downloaded += 1

    except Exception as e:

        failed += 1
        print(f"\nErreur : {filename}")
        print(e)

print("\n===================================")
print("download done")
print("===================================")
print(f"downloaded : {downloaded}")
print(f"ignore     : {skipped}")
print(f"Failed      : {failed}")
print(f"Folder     : {OUTPUT_DIR.resolve()}")

Obtention du token...
Authentification réussie.

505 produits trouvés dans le Metalink.



  0%|          | 0/505 [00:00<?, ?it/s]


Téléchargement terminé
Téléchargés : 505
Ignorés     : 0
Échecs      : 0
Dossier     : C:\Users\LENOVO\downloader\outputs\netcdf\sarah3


In [1]:
git --version

NameError: name 'git' is not defined